## HMM Model to infer the distribution of the read depth of a sample and its potential contaminant

The model has the following likelihood:

$$

\sum_{i=1}^{L} \mathrm{P}(N_A^i, N_T^i, N_C^i, N_G^i | n_1^i, n_2^i, \pi, \epsilon)

$$

with:
- L the length of the sequence (or only the variable sites taken into account? atm it is all the sites)
- i the current position, $i \in [1, L]$
- $N_A^i$ the number of A reads at position i
- $N_T^i$ the number of T reads at position i
- $N_C^i$ the number of C reads at position i
- $N_G^i$ the number of G reads at position i
- $n_1^i$ the nucleotide of the reference genome at position i
- $n_2^i$ the nucleotide of the contaminant genome at position i
- $\pi$ the proportion of the reference genome in the sample
- $\epsilon$ the sequencing error rate


In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.optimize import minimize_scalar
from scipy.special import logsumexp

In [46]:
# Retrieve a test sequence

qc_df = pd.read_table("../Existing_methods/viridian_example_data\DRR413445_vdn.v1.0.0\qc.tsv", sep="\t", header=0)
qc_df.iloc[:, 8:] = qc_df.iloc[:, 8:].replace(".", "0")
for col in qc_df.columns:
    if qc_df[col].dtype == "object":
        try:
            qc_df[col] = qc_df[col].astype(int)
        except ValueError:
            pass

qc_df["total_A"] = qc_df["A"] + qc_df["a"]
qc_df["total_C"] = qc_df["C"] + qc_df["c"]
qc_df["total_G"] = qc_df["G"] + qc_df["g"]
qc_df["total_T"] = qc_df["T"] + qc_df["t"]
qc_df["total_I"] = qc_df["I"] + qc_df["i"]
qc_df["total_D"] = qc_df["D"] + qc_df["d"]

# Calculate the proportion of all the nucleotides
qc_df["prop_A"] = qc_df["total_A"] / qc_df["Clean_depth"]
qc_df["prop_C"] = qc_df["total_C"] / qc_df["Clean_depth"]
qc_df["prop_G"] = qc_df["total_G"] / qc_df["Clean_depth"]
qc_df["prop_T"] = qc_df["total_T"] / qc_df["Clean_depth"]
qc_df["prop_I"] = qc_df["total_I"] / qc_df["Clean_depth"]
qc_df["prop_D"] = qc_df["total_D"] / qc_df["Clean_depth"]

# Remove the starting rows until clean_depth > 0
qc_df = qc_df[qc_df["Clean_depth"] > 0]


In [47]:
# df has columns: 'Ref_pos', 'Cons_nt', 'total_A', 'total_T', 'total_C', 'total_G'

def estimate_epsilon(df, threshold=0.95):
    counts = df[['total_A', 'total_T', 'total_C', 'total_G']]
    total = counts.sum(axis=1)
    max_nt = counts.max(axis=1)
    
    # Fraction of reads that match the dominant base
    max_frac = max_nt / total
    
    # Positions with high agreement
    high_confidence = max_frac > threshold
    error_reads = total[high_confidence] - max_nt[high_confidence]
    total_reads = total[high_confidence]
    
    # Estimate epsilon as total mismatches / total bases
    epsilon = error_reads.sum() / total_reads.sum()
    return epsilon

estimated_epsilon = estimate_epsilon(qc_df, threshold=0.95)
print(f"Estimated epsilon: {estimated_epsilon:.4f}")

Estimated epsilon: 0.0016


In [ ]:
def get_base_index(base):
    return {'A': 0, 'T': 1, 'C': 2, 'G': 3}[base]

def generate_candidate_pairs(consensus_base):
    bases = ['A', 'T', 'C', 'G']
    candidates = []
    for b in bases:
        if b != consensus_base:
            candidates.append((consensus_base, b))
            candidates.append((b, consensus_base))
    candidates.append((consensus_base, consensus_base))  # Include no-contamination case
    return candidates

def get_likelihoods_per_position(row, epsilon, candidate_pairs):
    counts = np.array([row['total_A'], row['total_T'], row['total_C'], row['total_G']])
    likelihoods = []

    for n1, n2 in candidate_pairs:
        # Compute probabilities
        p_ref = np.full(4, epsilon / 3)
        p_cont = np.full(4, epsilon / 3)
        p_ref[get_base_index(n1)] = 1 - epsilon
        p_cont[get_base_index(n2)] = 1 - epsilon

        best_pi = None
        best_ll = -np.inf

        for pi in np.linspace(0.01, 0.99, 20):
            p_mix = pi * p_ref + (1 - pi) * p_cont
            log_prob = np.sum(counts * np.log(p_mix + 1e-12))
            if log_prob > best_ll:
                best_ll = log_prob
                best_pi = pi

        likelihoods.append({
            'n1': n1, 'n2': n2,
            'pi': best_pi,
            'logL': best_ll
        })

    return sorted(likelihoods, key=lambda x: x['logL'], reverse=True)[0]

def analyze_all_sites(df, epsilon):
    consensus = df['Cons_nt']
    results = []
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Analyzing sites"):
        consensus_base = row['Cons_nt']
        if consensus_base not in ['A', 'T', 'C', 'G']:
            continue
        candidate_pairs = generate_candidate_pairs(consensus_base)
        best = get_likelihoods_per_position(row, epsilon, candidate_pairs)
        results.append(best)

    return pd.DataFrame(results)


In [49]:
analyzed_df = analyze_all_sites(qc_df, estimated_epsilon)

Analyzing sites: 100%|██████████| 29782/29782 [00:14<00:00, 2053.09it/s]


In [50]:
# Plot the evolution of the proportion of the reads of the major sample (column pi)

# plt.figure(figsize=(10, 6))
# sns.histplot(analyzed_df['pi'], bins=50, kde=True)
# plt.title('Distribution of Proportion of Reads (pi)')
# plt.xlabel('Proportion of Reads (pi)')
# plt.ylabel('Frequency')
# plt.grid()
# plt.show()


In [51]:
# Let's build a synthetic qc_df to test the model

import random

def simulate_reads(cons_seq, cont_seq=None, depth=100, epsilon=0.01, contamination_pi=1.0):
    """
    Simulate read counts at each position based on a consensus sequence,
    optionally mixed with a contaminant genome.
    
    Parameters:
        cons_seq (str): Consensus sequence (reference genome).
        cont_seq (str or None): Contaminant genome. If None, no contamination is simulated.
        depth (int): Number of reads per site.
        epsilon (float): Sequencing error rate.
        contamination_pi (float): Proportion of the main genome in the mixture (1.0 = pure).
        
    Returns:
        pd.DataFrame: DataFrame with simulated read counts and consensus base.
    """
    assert cont_seq is None or len(cons_seq) == len(cont_seq), "Sequences must be same length"

    bases = ['A', 'T', 'C', 'G']
    rows = []

    for i in range(len(cons_seq)):
        ref_base = cons_seq[i]
        if cont_seq:
            cont_base = cont_seq[i]
        else:
            cont_base = None
        
        # Create probability distributions
        ref_probs = np.full(4, epsilon / 3)
        cont_probs = np.full(4, epsilon / 3)
        
        ref_probs[bases.index(ref_base)] = 1 - epsilon
        if cont_base:
            cont_probs[bases.index(cont_base)] = 1 - epsilon

        # Mixture model: weighted sum of ref and contaminant
        if cont_base:
            mix_probs = contamination_pi * ref_probs + (1 - contamination_pi) * cont_probs
        else:
            mix_probs = ref_probs

        # Simulate multinomial reads from the mixed distribution
        counts = np.random.multinomial(depth, mix_probs)

        rows.append({
            'Cons_nt': ref_base,
            'total_A': counts[0],
            'total_T': counts[1],
            'total_C': counts[2],
            'total_G': counts[3]
        })

    return pd.DataFrame(rows)


In [66]:
# Simulate two sequences
np.random.seed(42)
length = 29000
depth = 800
ref_seq = ''.join(np.random.choice(['A', 'T', 'C', 'G'], length))
cont_seq = list(ref_seq)
# Introduce ~5% variation in the contaminant genome
for i in range(length):
    if np.random.rand() < 0.05:
        cont_seq[i] = random.choice([b for b in ['A', 'T', 'C', 'G'] if b != cont_seq[i]])
cont_seq = ''.join(cont_seq)

# Generate synthetic contaminated reads
synthetic_df = simulate_reads(ref_seq, cont_seq, depth=depth, epsilon=0.01, contamination_pi=0.82)
synthetic_df.head()


,Cons_nt,total_A,total_T,total_C,total_G
0,C,4,1,792,3
1,G,2,4,4,790
2,A,652,5,2,141
3,C,2,1,789,8
4,C,145,2,651,2


In [67]:
estimated_epsilon = estimate_epsilon(synthetic_df, threshold=0.95)

# Test the model on the synthetic data
analyzed_synthetic_df = analyze_all_sites(synthetic_df, estimated_epsilon)
analyzed_synthetic_df.head()

Analyzing sites:   0%|          | 0/29000 [00:00<?, ?it/s]

Analyzing sites: 100%|██████████| 29000/29000 [00:11<00:00, 2463.75it/s]


,n1,n2,pi,logL
0,C,C,0.95,-53.590196
1,G,G,0.95,-64.986109
2,A,G,0.95,-494.910995
3,C,C,0.95,-70.684065
4,C,A,0.95,-489.501912


In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

def get_base_index(base):
    return {'A': 0, 'T': 1, 'C': 2, 'G': 3}[base]

def generate_candidate_pairs(consensus_base):
    bases = ['A', 'T', 'C', 'G']
    candidates = []
    for b in bases:
        if b != consensus_base:
            candidates.append((consensus_base, b))
            candidates.append((b, consensus_base))
    candidates.append((consensus_base, consensus_base))  # Include no-contamination case
    return candidates

def analyze_all_sites_with_fixed_pi(df, epsilon, pi):
    results = []
    total_logL = 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Evaluating for pi={pi:.2f}"):
        consensus_base = row['Cons_nt']
        if consensus_base not in ['A', 'T', 'C', 'G']:
            continue

        candidate_pairs = generate_candidate_pairs(consensus_base)
        counts = np.array([row['total_A'], row['total_T'], row['total_C'], row['total_G']])

        best_logL = -np.inf
        best_n1, best_n2 = None, None

        for n1, n2 in candidate_pairs:
            p_ref = np.full(4, epsilon / 3)
            p_cont = np.full(4, epsilon / 3)
            p_ref[get_base_index(n1)] = 1 - epsilon
            p_cont[get_base_index(n2)] = 1 - epsilon
            p_mix = pi * p_ref + (1 - pi) * p_cont

            logL = np.sum(counts * np.log(p_mix + 1e-12))
            if logL > best_logL:
                best_logL = logL
                best_n1, best_n2 = n1, n2

        total_logL += best_logL
        results.append({
            'n1': best_n1,
            'n2': best_n2,
            'logL': best_logL
        })

    return total_logL, pd.DataFrame(results)

def estimate_best_pi(df, epsilon, pi_grid=np.linspace(0.60, 1, 9)):
    best_pi = None
    best_logL = -np.inf
    best_result_df = None

    for pi in pi_grid:
        total_logL, result_df = analyze_all_sites_with_fixed_pi(df, epsilon, pi)
        if total_logL > best_logL:
            best_logL = total_logL
            best_pi = pi
            best_result_df = result_df

    return best_pi, best_logL, best_result_df

In [ ]:
best_pi, best_logL, best_result_df = estimate_best_pi(synthetic_df, estimated_epsilon)

best_result_df, best_pi, best_logL

Evaluating for pi=1.00: 100%|██████████| 29000/29000 [00:19<00:00, 1483.98it/s]


,n1,n2,logL
0,C,C,-53.590196
1,G,G,-64.986109
2,A,G,-417.887885
3,C,C,-70.684065
4,C,A,-406.956173
...,...,...,...
28995,G,A,-363.033430
28996,G,A,-426.091566
28997,G,G,-30.798371
28998,G,G,-25.100415


In [71]:
best_pi_real, best_logL_real, best_result_df_real = estimate_best_pi(qc_df, estimated_epsilon)

Evaluating for pi=0.60:   0%|          | 0/29782 [00:00<?, ?it/s]

Evaluating for pi=1.00: 100%|██████████| 29782/29782 [00:17<00:00, 1690.78it/s]


In [72]:
best_result_df_real, best_pi_real, best_logL_real

(      n1 n2       logL
 0      A  A -14.485140
 1      G  G  -8.847233
 2      A  A -20.253154
 3      T  T  -8.867249
 4      C  C  -8.897274
 ...   .. ..        ...
 29777  T  T  -5.084156
 29778  A  A  -5.064140
 29779  T  T  -5.064140
 29780  C  C -10.752088
 29781  C  C -16.420020
 
 [29782 rows x 3 columns],
 np.float64(0.95),
 np.float64(-663999.4914597447))